<a href="https://colab.research.google.com/github/ashokmulchandani/CUDA-GPU-Colab-Computer-Vision-Project-Ashok-1/blob/main/1_Ashok_CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Check NVIDIA GPU
!nvidia-smi


Tue May 19 02:56:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: Check CUDA compiler version
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
# Cell 3: Check GPU details
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


In [4]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv


name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB


In [7]:
!nvcc device_query.cu -o device_query && ./device_query


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Device: Tesla T4
Compute Capability: 7.5
SMs (Streaming Multiprocessors): 40
Max Threads per SM: 1024
Max Threads per Block: 1024
Warp Size: 32
Total Global Memory: 15.64 GB
Shared Memory per Block: 48 KB
Clock Rate: 1.59 GHz
Memory Clock: 5.00 GHz
Memory Bus Width: 256 bits


In [8]:
%%writefile hello_world.cu
#include <stdio.h>

// __global__ = this function runs on GPU, called from CPU
__global__ void hello() {
    printf("Hello from GPU! Thread %d in Block %d\n", threadIdx.x, blockIdx.x);
}

int main() {
    // Launch kernel: <<<num_blocks, threads_per_block>>>
    hello<<<2, 4>>>();  // 2 blocks, 4 threads each = 8 threads total

    // Wait for GPU to finish before CPU continues
    cudaDeviceSynchronize();

    printf("Hello from CPU!\n");
    return 0;
}


Writing hello_world.cu


In [9]:
!nvcc hello_world.cu -o hello_world && ./hello_world


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Hello from GPU! Thread 0 in Block 0
Hello from GPU! Thread 1 in Block 0
Hello from GPU! Thread 2 in Block 0
Hello from GPU! Thread 3 in Block 0
Hello from GPU! Thread 0 in Block 1
Hello from GPU! Thread 1 in Block 1
Hello from GPU! Thread 2 in Block 1
Hello from GPU! Thread 3 in Block 1
Hello from CPU!


In [1]:
%%writefile hello_world2.cu
#include <stdio.h>

__global__ void hello() {
    // Calculate unique global thread ID
    int globalId = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Thread %d (Block %d, Local %d)\n", globalId, blockIdx.x, threadIdx.x);
}

int main() {
    hello<<<4, 8>>>();  // 4 blocks × 8 threads = 32 threads (1 full warp!)
    cudaDeviceSynchronize();
    printf("Total threads launched: 32\n");
    return 0;
}


Writing hello_world2.cu


In [2]:
!nvcc hello_world2.cu -o hello_world2 && ./hello_world2


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Thread 16 (Block 2, Local 0)
Thread 17 (Block 2, Local 1)
Thread 18 (Block 2, Local 2)
Thread 19 (Block 2, Local 3)
Thread 20 (Block 2, Local 4)
Thread 21 (Block 2, Local 5)
Thread 22 (Block 2, Local 6)
Thread 23 (Block 2, Local 7)
Thread 8 (Block 1, Local 0)
Thread 9 (Block 1, Local 1)
Thread 10 (Block 1, Local 2)
Thread 11 (Block 1, Local 3)
Thread 12 (Block 1, Local 4)
Thread 13 (Block 1, Local 5)
Thread 14 (Block 1, Local 6)
Thread 15 (Block 1, Local 7)
Thread 24 (Block 3, Local 0)
Thread 25 (Block 3, Local 1)
Thread 26 (Block 3, Local 2)
Thread 27 (Block 3, Local 3)
Thread 28 (Block 3, Local 4)
Thread 29 (Block 3, Local 5)
Thread 30 (Block 3, Local 6)
Thread 31 (Block 3, Local 7)
Thread 0 (Block 0, Local 0)
Thread 1 (Block 0, Local 1)
Thread 2 (Block 0, Local 2)
Thread 3 (Block 0, Local 3)
Thread 

In [4]:
%%writefile vector_add.cu
#include <stdio.h>

// GPU kernel: each thread adds one element
__global__ void vectorAdd(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {          // guard: don't go out of bounds
        c[i] = a[i] + b[i];
    }
}

int main() {
    int n = 10;  // small array for now
    int size = n * sizeof(int);

    // --- CPU (Host) arrays ---
    int h_a[] = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10};
    int h_b[] = {10, 20, 30, 40, 50, 60, 70, 80, 90, 100};
    int h_c[10];  // result

    // --- GPU (Device) arrays ---
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);    // allocate on GPU
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // --- Copy data: CPU → GPU ---
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // --- Launch kernel ---
    int threadsPerBlock = 4;
    int blocks = (n + threadsPerBlock - 1) / threadsPerBlock;  // ceiling division
    printf("Launching %d blocks × %d threads = %d threads\n", blocks, threadsPerBlock, blocks * threadsPerBlock);

    vectorAdd<<<blocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // --- Copy result: GPU → CPU ---
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // --- Print results ---
    for (int i = 0; i < n; i++) {
        printf("%d + %d = %d\n", h_a[i], h_b[i], h_c[i]);
    }

    // --- Free GPU memory ---
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing vector_add.cu


In [5]:
!nvcc vector_add.cu -o vector_add && ./vector_add


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Launching 3 blocks × 4 threads = 12 threads
1 + 10 = 11
2 + 20 = 22
3 + 30 = 33
4 + 40 = 44
5 + 50 = 55
6 + 60 = 66
7 + 70 = 77
8 + 80 = 88
9 + 90 = 99
10 + 100 = 110
